# Paired Edit Inference

Run single-image paired-edit inference in Colab using a `model_*.pkl` pix2pix-turbo checkpoint.

## Upload Location

- Put input images in `/content/drive/MyDrive/paired_edit_inputs/`
- Results will be written to `/content/drive/MyDrive/paired_edit_inference_outputs/`
- Checkpoints are read from `/content/drive/MyDrive/nail-retouch-paired-outputs/checkpoints/`
- If `INPUT_IMAGE_OVERRIDE` is `None`, the notebook auto-picks the newest file from the input folder
- If `CHECKPOINT_OVERRIDE` is `None`, the notebook auto-picks the newest `model_*.pkl` checkpoint
- The notebook enables low-VRAM inference by default and caps the input longest side at `1024`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import shutil
import subprocess

PROJECT_ROOT = Path('/content/nail-retouch-assistant')
UPSTREAM_ROOT = Path('/content/img2img-turbo')

if PROJECT_ROOT.exists():
    shutil.rmtree(PROJECT_ROOT)
subprocess.run(
    ['git', 'clone', 'https://github.com/Zifpen/nail-retouch-assistant.git', str(PROJECT_ROOT)],
    check=True,
)

if UPSTREAM_ROOT.exists():
    shutil.rmtree(UPSTREAM_ROOT)
subprocess.run(
    ['git', 'clone', 'https://github.com/GaParmar/img2img-turbo.git', str(UPSTREAM_ROOT)],
    check=True,
)

subprocess.run(
    ['python', '-m', 'pip', 'install', '-q', 'diffusers', 'transformers', 'accelerate', 'peft', 'sentencepiece'],
    check=True,
)

print('repo head:', subprocess.check_output(['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'HEAD']).decode().strip())


In [ ]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive')
INPUT_DIR = DRIVE_ROOT / 'paired_edit_inputs'
OUTPUT_DIR = DRIVE_ROOT / 'paired_edit_inference_outputs'
CHECKPOINT_DIR = DRIVE_ROOT / 'nail-retouch-paired-outputs' / 'checkpoints'

# Optional overrides.
INPUT_IMAGE_OVERRIDE = None
CHECKPOINT_OVERRIDE = None
PROMPT_OVERRIDE = None
MAX_SIDE_OVERRIDE = 1024
LOW_VRAM = True

DEFAULT_PROMPT = (
    'professional manicure photo retouch, clean cuticles, clean sidewalls, '
    'refined nail shape, glossy nail surface, natural skin texture, realistic hand photo'
)

INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
assert CHECKPOINT_DIR.exists(), f'Missing checkpoint dir: {CHECKPOINT_DIR}'

input_candidates = sorted(
    [p for p in INPUT_DIR.glob('*') if p.is_file()],
    key=lambda p: p.stat().st_mtime,
)
assert input_candidates or INPUT_IMAGE_OVERRIDE, f'No input images found in {INPUT_DIR}'

checkpoint_candidates = sorted(
    CHECKPOINT_DIR.glob('model_*.pkl'),
    key=lambda p: int(p.stem.split('_')[-1]),
)
assert checkpoint_candidates, f'No model_*.pkl found in {CHECKPOINT_DIR}'

INPUT_IMAGE = Path(INPUT_IMAGE_OVERRIDE) if INPUT_IMAGE_OVERRIDE else input_candidates[-1]
CHECKPOINT_PATH = Path(CHECKPOINT_OVERRIDE) if CHECKPOINT_OVERRIDE else checkpoint_candidates[-1]
PROMPT = PROMPT_OVERRIDE or DEFAULT_PROMPT

print('upload input images to:', INPUT_DIR)
print('selected input:', INPUT_IMAGE)
print('selected checkpoint:', CHECKPOINT_PATH)
print('output dir:', OUTPUT_DIR)
print('prompt:', PROMPT)
print('max side:', MAX_SIDE_OVERRIDE)
print('low vram:', LOW_VRAM)


In [ ]:
cmd = [
    'python',
    str(PROJECT_ROOT / 'src' / 'paired_edit' / 'run_paired_edit_inference.py'),
    '--input',
    str(INPUT_IMAGE),
    '--checkpoint',
    str(CHECKPOINT_PATH),
    '--output-dir',
    str(OUTPUT_DIR),
    '--upstream-dir',
    str(UPSTREAM_ROOT),
    '--prompt',
    PROMPT,
]
if LOW_VRAM:
    cmd.append('--low-vram')
if MAX_SIDE_OVERRIDE is not None:
    cmd.extend(['--max-side', str(MAX_SIDE_OVERRIDE)])
print(' '.join(cmd))
result = subprocess.run(cmd, text=True, capture_output=True)
if result.stdout:
    print(result.stdout)
if result.stderr:
    print(result.stderr)
result.check_returncode()


In [ ]:
from pathlib import Path
from IPython.display import display
from PIL import Image

output_path = OUTPUT_DIR / f'{INPUT_IMAGE.stem}_{CHECKPOINT_PATH.stem}.png'
metadata_path = output_path.with_suffix('.json')

print('saved output:', output_path)
print('saved metadata:', metadata_path)

display(Image.open(output_path))
